### Step 1: Build series lookup from the Index sheet
The index sheet mapes every Series ID to a respective category name and type. Creating a lookup allows the analysis to filter and label columns of data points

In [30]:
import pandas as pd
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'

# Create dataframe from raw excel file
df_index = pd.read_excel(DATA_RAW / '850101.xlsx', sheet_name = 'Index')

# Keep only relevant columns and rows
meta = df_index.iloc[10:, [0, 3, 4]].copy()

# Label columns
meta.columns = ['description', 'series_type', 'series_id']

# Drop with no series_id
meta = meta.dropna(subset=['series_id'])

'''accepts description input as str to output series cateogry as str'''
def extract_category(desc):
    parts = [p.strip() for p in str(desc).split(';')]
    return parts[2].strip() if len(parts) > 2 else desc

# Create new column containing cateogry of series based on description
meta['category'] = meta['description'].apply(extract_category)

# Store look up data
lookup = meta.set_index('series_id')[['category', 'series_type']]


### Step 2: Read data

In [31]:
# Read data sheet
df_data = pd.read_excel(DATA_RAW / '850101.xlsx', sheet_name = 'Data1', header=9, index_col=0)

# Ensure that the first column is datetime object
df_data.index = pd.to_datetime(df_data.index, errors='coerce')
df_data = df_data[df_data.index.notna()]

# Label first/index column as date
df_data.index.name = 'date'

# Drop rows with no date
df_data = df_data[df_data.index.notna()]
df_data.head()



,A3348591K,A3348600A,A3348609W,A3348618X,A3348627A,A3348636C,A3348582J,A3348594T,A3348603J,A3348612K,...,A3348630R,A3348639K,A3348585R,A3348597X,A3348606R,A3348615T,A3348624V,A3348633W,A3348642X,A3348588W
date,,,,,,,,,,,,,,,,,,,,,
1982-04-01,1162.6,592.3,359.9,460.1,479.1,342.4,3396.4,1166.8,653.4,360.6,...,507.8,349.7,3518.7,1172.8,652.8,362.6,483.1,505.0,347.5,3523.4
1982-05-01,1150.9,629.6,386.6,502.6,486.1,342.1,3497.9,1178.2,648.7,362.5,...,502.2,346.3,3527.6,1181.3,654.1,361.9,484.8,504.8,346.3,3533.6
1982-06-01,1160.0,607.4,350.5,443.8,467.5,328.7,3357.8,1203.4,655.7,365.1,...,506.8,350.8,3561.5,1192.4,655.7,361.8,487.0,504.6,345.8,3547.0
1982-07-01,1206.4,632.4,359.3,459.1,491.1,338.5,3486.8,1209.5,660.5,361.9,...,503.6,341.5,3553.9,1202.8,656.6,361.5,489.1,505.3,345.4,3560.6
1982-08-01,1152.5,622.6,325.2,438.4,485.7,331.5,3355.9,1198.2,659.8,359.2,...,505.9,342.6,3581.8,1213.2,656.5,361.9,490.3,505.5,346.3,3573.6


### Step 3: Pivot data

In [32]:
df_long = df_data.reset_index().melt(
    id_vars=['date'],
    var_name='series_id',
    value_name='turnover_m')

df_long.head()

,date,series_id,turnover_m
0,1982-04-01,A3348591K,1162.6
1,1982-05-01,A3348591K,1150.9
2,1982-06-01,A3348591K,1160.0
3,1982-07-01,A3348591K,1206.4
4,1982-08-01,A3348591K,1152.5


### Step 4: Join index and data 

The cateogry and series type information will be added to the data 

The data will be cut down to only Original data, as these are the raw numbers as collected.

Other series types Seasonally Adjusted and Trend have been analyzed and transformed by the ABS and should not be included so as not to distort my own analysis


In [33]:
df_long = df_long.join(lookup, on='series_id')
df_clean = df_long[df_long['series_type'] == 'Original'].copy()

### Step 5: Add helper data columns
Helper columns such as year, month, and month name make filtering and grouping in SQL and Power BI much easier later

In [34]:
df_clean['year'] = df_clean['date'].dt.year 
df_clean['month'] = df_clean['date'].dt.month 
df_clean['month_name'] = df_clean['date'].dt.strftime('%b')

### Step 6: Save data
Sort data to read chronologically within each category for reading ease

Save data into new, cleaned csv

In [35]:
df_clean = df_clean[['date', 'year', 'month', 'month_name', 'category', 'turnover_m']]
df_clean = df_clean.sort_values(['category', 'date']).reset_index(drop=True)

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df_clean.to_csv(DATA_PROCESSED / 'retail_clean.csv', index=False)

print('Done. Saved: retail_clean.csv')

Done. Saved: retail_clean.csv


### Step 7: Load data into sql database

In [36]:
import sqlite3
DATA_DB = PROJECT_ROOT / 'data' / 'database'

# Read clean CSV created in 01_cleaning.ipynb
df = pd.read_csv(DATA_PROCESSED / 'retail_clean.csv')

# Convert string data to datetime objects
df['date'] = pd.to_datetime(df['date'])

# Create / opens databse file
conn = sqlite3.connect(DATA_DB / 'retail.db')

# Write dataframe into sql table
df.to_sql('retail_turnover', conn, if_exists='replace', index=False)

# Check if the table is populated
result = pd.read_sql('SELECT COUNT(*) as n FROM retail_turnover', conn)
print(result)

# Check if all expected categories are present
cats = pd.read_sql('SELECT DISTINCT category FROM retail_turnover ORDER BY category', conn)
print(cats)

# Close connection
conn.close()

print('Database created:', DATA_DB / 'retail.db')

      n
0  3633
                                            category
0      Cafes, restaurants and takeaway food services
1  Clothing, footwear and personal accessory reta...
2                                  Department stores
3                                     Food retailing
4                          Household goods retailing
5                                    Other retailing
6                                   Total (Industry)
Database created: /Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db
